# Model Evaluation – Baseline

Evaluate baseline Random Forest model performance 
and generate predictions with probability scores.

2. Load model + data

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ModelEvaluation") \
    .master("local[*]") \
    .getOrCreate()

train_df = spark.read.parquet("../data/processed/train")
test_df = spark.read.parquet("../data/processed/test")

print("Data loaded")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/13 16:26:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

Data loaded


3. Re-train model

In [2]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=20,
    maxDepth=5,
    seed=42
)

pipeline = Pipeline(stages=[rf])
model = pipeline.fit(train_df)

print("Model ready")

26/05/13 16:26:57 WARN MemoryStore: Not enough space to cache rdd_32_10 in memory! (computed 211.0 MiB so far)
26/05/13 16:26:57 WARN BlockManager: Persisting block rdd_32_10 to disk instead.
26/05/13 16:26:57 WARN MemoryStore: Not enough space to cache rdd_32_3 in memory! (computed 211.0 MiB so far)
26/05/13 16:26:57 WARN BlockManager: Persisting block rdd_32_3 to disk instead.
26/05/13 16:27:02 WARN MemoryStore: Not enough space to cache rdd_32_10 in memory! (computed 211.0 MiB so far)
26/05/13 16:27:03 WARN MemoryStore: Not enough space to cache rdd_32_3 in memory! (computed 211.0 MiB so far)
26/05/13 16:27:06 WARN MemoryStore: Not enough space to cache rdd_32_10 in memory! (computed 211.0 MiB so far)
26/05/13 16:27:06 WARN MemoryStore: Not enough space to cache rdd_32_3 in memory! (computed 211.0 MiB so far)
26/05/13 16:27:11 WARN MemoryStore: Not enough space to cache rdd_32_3 in memory! (computed 211.0 MiB so far)
26/05/13 16:27:11 WARN MemoryStore: Not enough space to cache rdd_

Model ready


4. Predictions for TEST

In [3]:
predictions = model.transform(test_df)

predictions.select("label", "prediction", "probability").show(5)

[Stage 21:===========================================>              (3 + 1) / 4]

+-----+----------+--------------------+
|label|prediction|         probability|
+-----+----------+--------------------+
|    0|       0.0|[0.99833061749303...|
|    0|       0.0|[0.99833061749303...|
|    0|       0.0|[0.99833061749303...|
|    0|       0.0|[0.99833061749303...|
|    0|       0.0|[0.99833061749303...|
+-----+----------+--------------------+
only showing top 5 rows



5. Confusion matrix

In [4]:
predictions.groupBy("label", "prediction").count().show()

[Stage 22:================================================>       (13 + 2) / 15]

+-----+----------+-------+
|label|prediction|  count|
+-----+----------+-------+
|    0|       0.0|1273080|
|    1|       0.0|     12|
|    1|       1.0|   1618|
+-----+----------+-------+



6. Metrics (Spark ML evaluator)

In [5]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator_precision = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

evaluator_recall = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
)

evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

precision = evaluator_precision.evaluate(predictions)
recall = evaluator_recall.evaluate(predictions)
f1 = evaluator_f1.evaluate(predictions)

print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

[Stage 29:================================================>       (13 + 2) / 15]

Precision: 0.9999905861828263
Recall: 0.999990586094092
F1-score: 0.9999905687260554


In [6]:
tp = predictions.filter((predictions.label == 1) & (predictions.prediction == 1)).count()
fn = predictions.filter((predictions.label == 1) & (predictions.prediction == 0)).count()

fraud_recall = tp / (tp + fn)

print("Fraud Recall:", fraud_recall)

Fraud Recall: 0.992638036809816


7. Save predictions

In [7]:
predictions.select("features", "label", "prediction", "probability") \
    .write.mode("overwrite") \
    .parquet("../data/processed/predictions")

## Results

The model generates predictions along with probability scores.

### Observations:
- high weighted metrics due to strong class imbalance
- fraud detection recall is high (~0.99)
- model predicts both classes correctly

### Important:
Due to class imbalance, weighted metrics are misleading.
Fraud-specific recall provides a more realistic evaluation.

Results may also be influenced by:
- strong feature correlations
- possible data leakage

### Conclusion:
Baseline model works, but requires deeper evaluation and improvement.